<a href="https://colab.research.google.com/github/maclandrol/cours-ia-med/blob/master/devoirs/Homework_Chest_XRay_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Devoir 2: Analyse de Radiographies Thoraciques

**Enseignant:** Emmanuel Noutahi, PhD

**Note:** /20 points

---

## Objectif

Analyser des radiographies thoraciques, tester la robustesse du modèle TorchXRayVision, et comprendre son comportement sur des pathologies hors distribution.

## Barème

1. Setup et fonctions (3 pts)
2. Analyse de 3 radiographies (6 pts)
3. Test de robustesse aux distorsions (6 pts)
4. Analyse OOD et rapport (5 pts)

## Partie 1: Setup (3 points)

In [ ]:
!pip install -q torchxrayvision torch torchvision scikit-image pillow matplotlib

In [ ]:
import torchxrayvision as xrv
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage import transform as sktransform, util, exposure, filters, io as skio
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

### Question 1.1 (1 pt): Charger le modèle

In [ ]:
#===========================
# TODO: Chargez le modèle DenseNet avec weights="densenet121-res224-all"

model = # Complétez
model = model.to(device)
model.eval()

#===========================
print(f"Pathologies: {len(model.pathologies)}")

### Question 1.2 (2 pts): Fonctions utilitaires

Complétez les deux fonctions principales.

In [ ]:
transform = torchvision.transforms.Compose([
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(224),
])

def prepare_image(img_input, from_file=True):
    #===========================
    # TODO: Complétez la fonction
    # 1. Charger l'image si from_file=True
    # 2. Normaliser: xrv.datasets.normalize(img, 255)
    # 3. Convertir en 2D si besoin
    # 4. Ajouter canal: img[None, :, :]
    # 5. Appliquer transform
    # 6. Convertir en tensor
    
    # Votre code ici
    
    #===========================
    return img_tensor

def predict(img_tensor, model):
    #===========================
    # TODO: Complétez la fonction
    # 1. Passer img_tensor au modèle
    # 2. Extraire les scores
    # 3. Trier par ordre décroissant
    
    # Votre code ici
    
    #===========================
    return scores, indices

## Partie 2: Analyse de 3 Radiographies (6 points)

### Question 2.1 (2 pts): Image normale

In [ ]:
url1 = "https://raw.githubusercontent.com/mlmed/torchxrayvision/master/tests/16747_3_1.jpg"

#===========================
# TODO: Chargez et analysez l'image
img1 = # Complétez
scores1, indices1 = # Complétez
#===========================

# Affichage
plt.figure(figsize=(6, 6))
plt.imshow(img1[0].cpu(), cmap='gray')
plt.title("Image 1")
plt.axis('off')
plt.show()

print("Top 5:")
for i in range(5):
    idx = indices1[i]
    print(f"{model.pathologies[idx]}: {scores1[idx]:.1%}")

### Question 2.2 (2 pts): Image avec pathologie

Trouvez et analysez une radiographie avec une pathologie visible.

In [ ]:
#===========================
# TODO: Uploadez ou chargez une image avec pathologie
# Analysez-la et affichez les résultats

# Votre code ici

#===========================

### Question 2.3 (2 pts): Image COVID (OOD)

In [ ]:
!wget -q https://raw.githubusercontent.com/ieee8023/covid-chestxray-dataset/master/images/01E392EE-69F9-4E33-BFCE-E5C968654078.jpeg -O covid.jpg

#===========================
# TODO: Chargez et analysez l'image COVID
img_covid = # Complétez
scores_covid, indices_covid = # Complétez
#===========================

plt.figure(figsize=(6, 6))
plt.imshow(img_covid[0].cpu(), cmap='gray')
plt.title("COVID-19 (OOD)")
plt.axis('off')
plt.show()

print("Top 5:")
for i in range(5):
    idx = indices_covid[i]
    print(f"{model.pathologies[idx]}: {scores_covid[idx]:.1%}")

## Partie 3: Test de Robustesse (6 points)

### Question 3.1 (2 pts): Appliquer 5 distorsions

Choisissez 5 distorsions parmi: rotation, bruit, contraste, gamma, blur, zoom

In [ ]:
# Fonctions de distorsion (fournies)
def ensure_grayscale_2d(img):
    img = np.asarray(img)
    if img.ndim == 2:
        return img
    if img.ndim == 3:
        return img[..., :3].mean(axis=2)
    raise ValueError(f"Format non supporté")

def rotate_image(img, angle):
    img = ensure_grayscale_2d(img).astype(np.float32)
    return sktransform.rotate(img, angle, mode='edge', preserve_range=True).astype(np.float32)

def gaussian_noise_image(img, var=0.5):
    img = ensure_grayscale_2d(img).astype(np.float32)
    img01 = img / 255.0
    noisy = util.random_noise(img01, mode='gaussian', var=var)
    return (noisy * 255.0).astype(np.float32)

def low_contrast_image(img, factor=0.5):
    img = ensure_grayscale_2d(img).astype(np.float32)
    mean_val = img.mean()
    return np.clip((img - mean_val) * factor + mean_val, 0, 255).astype(np.float32)

def gamma_bright_image(img, gamma=0.7):
    img = ensure_grayscale_2d(img).astype(np.float32)
    img01 = np.clip(img / 255.0, 0, 1)
    out = exposure.adjust_gamma(img01, gamma=gamma)
    return (out * 255.0).astype(np.float32)

def gaussian_blur_image(img, sigma=1.5):
    img = ensure_grayscale_2d(img).astype(np.float32)
    return filters.gaussian(img, sigma=sigma, preserve_range=True).astype(np.float32)

In [ ]:
#===========================
# TODO: Appliquez 5 distorsions différentes à img1
# Créez un dictionnaire distortions = {"Nom": tensor, ...}

img_raw = skio.imread(url1)

distortions = {
    "Original": prepare_image(img_raw, from_file=False),
    # Ajoutez 5 distorsions ici
    # Exemple: "Rotation 45°": prepare_image(rotate_image(img_raw, 45), from_file=False),
}

#===========================
print(f"Distorsions créées: {len(distortions)}")

### Question 3.2 (2 pts): Visualiser les distorsions

In [ ]:
#===========================
# TODO: Créez un subplot montrant les 6 images (1 originale + 5 distorsions)
# Format: 2 lignes x 3 colonnes

# Votre code ici

#===========================

### Question 3.3 (2 pts): Analyser l'impact

Pour chaque distorsion:
1. Faites une prédiction
2. Comparez avec l'original
3. Créez un tableau montrant comment les scores changent

In [ ]:
#===========================
# TODO: Analysez chaque distorsion
# Créez un tableau comparatif

results = {}

for name, img_tensor in distortions.items():
    # Votre code ici
    pass

#===========================

# Affichage
print("\nIMPACT DES DISTORSIONS")
print("="*70)
# Votre code d'affichage ici

## Partie 4: Analyse OOD et Rapport (5 points)

### Question 4.1 (2 pts): Comparaison COVID vs Normal

In [ ]:
#===========================
# TODO: Créez un graphique comparant:
# - Scores des 8 pathologies top pour l'image normale
# - Scores des 8 pathologies top pour l'image COVID
# Format: barres groupées

# Votre code ici

#===========================

### Question 4.2 (3 pts): Rapport Final

Rédigez un rapport structuré (10-15 lignes) couvrant:

1. **Résultats principaux** (3-4 lignes): Que montrent les 3 images analysées?

2. **Robustesse** (3-4 lignes): Quelles distorsions impactent le plus? Pourquoi?

3. **Comportement OOD** (2-3 lignes): Comment le modèle réagit au COVID?

4. **Recommandations cliniques** (2-3 lignes): Comment utiliser cet outil en pratique?

**Votre rapport:**

[Écrivez ici]

## Remise

Avant de soumettre:
1. Exécutez: Exécution → Tout exécuter
2. Vérifiez qu'il n'y a pas d'erreurs
3. Sauvegardez votre notebook
4. Partagez le lien ou envoyez par email

**Bon travail!**